# Четыре семейства, Optuna и дополнительные эксперименты
Протокол до расчётов: 3 внешних групповых стратифицированных блока × 3 повторения; 3 внутренних блока. 50 попыток на семейство в каждом внешнем обучении и финальном поиске. Критерий — средний AP смерти; равенство 1e-6 разрешается Brier, затем LR→RF→CatBoost→XGBoost. Внешняя оценка всей процедуры сохраняется отдельно от оценок семейств.

Обработка включена в поиск (все строки / веса / стратифицированный undersampling; индикаторы пропусков да/нет). Контролируемые условные эксперименты выбранного семейства: по 20 попыток на каждую фиксированную ветвь в каждом внешнем обучении. Этот отдельный меньший бюджет зафиксирован заранее для вычислительной выполнимости; он одинаков для всех ветвей и не меняет финальное правило. Эти исследования не являются независимой оценкой выбора семейства. Случайный undersampling — дополнительный контроль. Все три варианта ТАГ/МС настраиваются заново: полный набор, без обоих, только ТАГ, только МС (4 согласованные версии). Финальное правило сохраняет полный набор независимо от этих результатов.

Новый эксперимент: отдельный индикатор отсутствия СРБ исключён; значение СРБ, обучение импутации, все остальные признаки и разрешённые индикаторы сохранены. Когорта, seeds, разбиения, пространства и бюджеты поиска прежние. Старый storage не используется.

## Фактическая конфигурация нового запуска

RUN_ID `oks_no_crp_indicator_v2_d138be90f9f1`. По прежнему внутреннему критерию выбрана CatBoost: `{"strategy": "stratified_4to1", "indicators": true, "params": {"iterations": 200, "depth": 6, "learning_rate": 0.04849866187746487, "l2_leaf_reg": 35.613541610725896}}`. Полная входная когорта — 906 записей; классификатор обучен на 215 реальных записях после выбранной обработки дисбаланса. Индикаторы — по всем включённым признакам, кроме СРБ. Отчёт 02 завершён и уточнён этапом 03 на основании текущих сохранённых результатов.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "studies/acs/configs/layout.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри клонированного acs-research-projects")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from acs_research.paths import study_paths
P = study_paths(PROJECT_ROOT)
STUDY = P.study
OUT = P.run
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in
            ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
print(platform.python_version(), "seed", BASE_SEED, VERSIONS)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def figsave(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight")
    plt.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close()


In [ ]:
import time, warnings, joblib, optuna, sqlite3, math
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix, roc_curve, precision_score, recall_score, f1_score, accuracy_score
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits
threadpool_limits(2)
optuna.logging.set_verbosity(optuna.logging.WARNING)
meta=json.loads((OUT/"01_конфигурация.json").read_text())
assert sha(OUT/"01_данные.csv") == meta["data_sha256"]
for p,h in meta["source_sha256"].items(): assert sha(PROJECT_ROOT/p)==h
D=pd.read_csv(OUT/"01_данные.csv")
FEATURES=meta["features"]
X=D[FEATURES];y=D.hospital_death.to_numpy();groups=D.group_id.to_numpy()
FAMILIES=["LR","RF","CatBoost","XGBoost"]
STRATEGIES=["all_real","class_weights","stratified_4to1"]
BASE={"LR":{"C":1.0},"RF":{"n_estimators":500,"max_depth":5,"min_samples_leaf":1,"max_features":"sqrt"},"CatBoost":{"iterations":300,"depth":4,"learning_rate":0.05,"l2_leaf_reg":3.0},"XGBoost":{"n_estimators":300,"max_depth":3,"learning_rate":0.05,"min_child_weight":1.0,"subsample":1.0,"colsample_bytree":1.0,"reg_lambda":1.0,"reg_alpha":0.0}}
CONFIG=json.loads((STUDY/"configs/training_protocol.json").read_text())
CONFIG.update(data_sha256=meta["data_sha256"], versions=VERSIONS)
code='\n'.join(c.source for c in __import__('nbformat').read(STUDY/"machine_learning/notebooks/02_модели_Optuna_и_эксперименты.ipynb",as_version=4).cells if c.cell_type=="code")
CONFIG["code_sha256"]=hashlib.sha256(code.encode()).hexdigest()
CONFIG["excluded_missing_indicators"]=["crp"]
CONFIG["module_sha256"]=sha(PROJECT_ROOT/"src/acs_research/modeling.py")
CONFIG["parent_run_id"]="oks_no_crp_indicator_v2_d138be90f9f1"
RUN_ID="acs_repository_v1_"+hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
CONFIG["run_id"]=RUN_ID
savejson(CONFIG,"02_протокол.json")
STORAGE="sqlite:///"+str(OUT/"02_optuna_без_индикатора_СРБ.sqlite")
def log(message):
    line=f"{datetime.datetime.now(datetime.timezone.utc).isoformat()} {RUN_ID} {message}"
    print(line,flush=True)
    with (OUT/"02_журнал_запуска.txt").open("a") as f:f.write(line+"\n")
log("Protocol fixed before outer computations")
SAMPLING_AUDIT=[];INNER_SPLITS=[]
def splits(a,b,g,seed):
    ss=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(a,b,g))
    for tr,te in ss:
        assert not set(g[tr])&set(g[te])
        assert len(np.unique(b[tr]))==len(np.unique(b[te]))==2
    return ss
OUTER=[];splitrows=[]
for repeat in range(3):
    for fold,(tr,te) in enumerate(splits(X,y,groups,BASE_SEED+repeat)):
        OUTER.append((repeat,fold,tr,te))
        for side,ids in [("train",tr),("test",te)]:
            splitrows.extend(dict(repeat=repeat,fold=fold,side=side,row_id=int(D.row_id.iloc[i]),group_id=int(groups[i])) for i in ids)
pd.DataFrame(splitrows).to_csv(OUT/"02_внешние_разбиения.csv",index=False)
display(pd.DataFrame([dict(repeat=r,fold=f,train_n=len(tr),train_deaths=int(y[tr].sum()),test_n=len(te),test_deaths=int(y[te].sum())) for r,f,tr,te in OUTER]))


## Обучение и защита от утечки
Числовые показатели импутируются медианой, бинарные/категориальные — модой текущего обучения, категории one-hot; LR масштабирует только непрерывные показатели. Пустой обучающий столбец сохраняется и заполняется 0. При выбранных индикаторах добавляются отдельные флаги для всех 15 признаков; неизменные флаги допустимы. Преобразования обучаются на реальном train до undersampling, никогда на валидации. Это заранее установленный вариант использования доступных обучающих данных. Killip остаётся категорией 1–4.

Стратифицированный отбор: медиана возраста текущего train; Killip 1–2 / 3–4 / неизвестен, неизвестный возраст — отдельная группа. Квоты большинства пропорциональны его стратам и распределяются методом наибольших остатков до ровно min(4×число смертей, число выживших), без возвращения. Все смерти сохраняются. Веса рассчитаны по текущему train; одновременно с отбором не используются.

Индикаторы строятся по всем включённым признакам, кроме СРБ. Проверки выполняются при каждом обучении, включая baseline; ветка без индикаторов не содержит никаких индикаторов.

In [ ]:
from acs_research import modeling
modeling.configure(globals())
from acs_research.modeling import (
    make_preprocessor, sample_indices, make_model, train, predict,
    threshold, metrics, suggest, inner_predictions, tune, choose, records)
FEATURE_AUDIT=[]
for f in FAMILIES:
    for indicator_option in [False,True]:
        for drop in [[],["tag","ms"],["ms"],["tag"]]:
            columns=[c for c in FEATURES if c not in drop]
            processor=make_preprocessor(columns,f,indicator_option)
            z_check=processor.fit_transform(X.iloc[OUTER[0][2]][columns])
            names=processor.get_feature_names_out().tolist()
            assert "numeric__crp" in names and "missing__missingindicator_crp" not in names
            assert len(names)==z_check.shape[1]
            assert sum(s.startswith("missing__") for s in names)==(len(columns)-1 if indicator_option else 0)
            FEATURE_AUDIT.append(dict(run_id=RUN_ID,family=f,indicators=indicator_option,dropped=json.dumps(drop),n_features=len(names),crp_value_present=True,crp_missing_flag_present=False,transformed_features=json.dumps(names)))
pd.DataFrame(FEATURE_AUDIT).to_csv(OUT/"02_проверка_состава_признаков.csv",index=False)
print("CRP value retained; missing-CRP flag absent in all families and baseline preprocessing")


## Вложенная оценка baseline, настроенных семейств и selector
Для каждого внешнего train создаётся общий inner split, заново выполняются четыре Optuna-поиска. Порог вычисляется по внутренним OOF выбранной конфигурации, максимум balanced accuracy; в равенстве выбирается большая чувствительность. Внешняя проверка сохраняет исходную частоту событий. Baseline — фиксированные параметры и без весов, с индикаторами; его порог также выбирается внутри train. Эффект настройки относится ко всему пайплайну.

In [ ]:
OOF=[];RESULTS=[];FOLD_METRICS=[]
for repeat,fold,tr,te in OUTER:
    seed=BASE_SEED+100*repeat+10*fold
    a=X.iloc[tr];b=y[tr];g=groups[tr];ss=splits(a,b,g,seed+1000)
    for k,(it,iv) in enumerate(ss):
        INNER_SPLITS.extend(dict(repeat=repeat,fold=fold,inner_fold=k,side=side,row_id=int(D.row_id.iloc[tr[j]]),group_id=int(g[j])) for side,ii in [("train",it),("validation",iv)] for j in ii)
    candidates=[]
    for family in FAMILIES:
        result=tune(a,b,g,family,ss,f"r{repeat}_f{fold}",seed)
        candidates.append(result);RESULTS.append(dict(repeat=repeat,fold=fold,**result))
        model=train(a,b,family,result["config"],seed)
        p=predict(model,X.iloc[te]);t=result["threshold"]
        OOF.extend(records(te,repeat,fold,family,p,t,result["config"]))
        FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=family,**metrics(y[te],p,t)))
        cfg={"params":BASE[family],"strategy":"all_real","indicators":True}
        pi,_,_=inner_predictions(a,b,g,family,cfg,ss,seed);tb=threshold(b,pi)
        baseline=train(a,b,family,cfg,seed);pb=predict(baseline,X.iloc[te])
        OOF.extend(records(te,repeat,fold,family+"_baseline",pb,tb,cfg))
        FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=family+"_baseline",**metrics(y[te],pb,tb)))
    winner=choose(candidates);family=winner["family"]
    model=train(a,b,family,winner["config"],seed);p=predict(model,X.iloc[te]);t=winner["threshold"]
    OOF.extend(records(te,repeat,fold,"selector",p,t,{"selected_family":family,**winner["config"]}))
    FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family="selector",selected_family=family,**metrics(y[te],p,t)))
    log(f"Outer r{repeat} f{fold} completed, selector={family}")
    pd.DataFrame(OOF).to_csv(OUT/"02_oof_predictions.csv",index=False)
    pd.DataFrame(FOLD_METRICS).to_csv(OUT/"02_метрики_блоков.csv",index=False)
    savejson(RESULTS,"02_результаты_поиска.json")
pd.DataFrame(INNER_SPLITS).to_csv(OUT/"02_внутренние_разбиения.csv",index=False)
pd.DataFrame(SAMPLING_AUDIT).to_csv(OUT/"02_аудит_балансировки.csv",index=False)
oof=pd.DataFrame(OOF);fm=pd.DataFrame(FOLD_METRICS)
assert oof.groupby(["repeat","family"]).row_id.nunique().eq(len(D)).all()
assert not oof.duplicated(["repeat","family","row_id"]).any()
display(fm.groupby("family")[["AP","ROC_AUC","Brier"]].agg(["mean","std","min","max"]))


## Финальный поиск на всей когорте
Семейство артефакта выбирается по внутреннему AP полного поиска, без выбора по внешним результатам. Метрики этого внутреннего поиска оптимистичны и не заменяют selector_oof. Финальная модель обучается на всей допустимой когорте.

In [ ]:
final_ss=splits(X,y,groups,BASE_SEED+9999)
FINAL_RESULTS=[tune(X,y,groups,f,final_ss,"final",BASE_SEED+9999) for f in FAMILIES]
FINAL=choose(FINAL_RESULTS);SELECTED_FAMILY=FINAL["family"]
artifact=train(X,y,SELECTED_FAMILY,FINAL["config"],BASE_SEED+9999)
artifact["threshold"]=FINAL["threshold"];artifact["source_data_sha256"]=meta["data_sha256"];artifact["versions"]=VERSIONS
joblib.dump(artifact,OUT/"02_final_model.joblib")
savejson({"run_id":RUN_ID,"selection":FINAL,"all_families":FINAL_RESULTS,"features":FEATURES,"interpretation":"full-cohort fitted artifact; independent quality belongs to nested selector"},"02_final_config.json")
print("FINAL",FINAL);display(pd.DataFrame(FINAL_RESULTS)[["family","inner_AP","inner_Brier","threshold","completed"]])


## Контролируемые условные эксперименты
Выбранное семейство фиксировано глобально; приведённые ниже оценки условны относительно этого выбора. Каждый вариант заново настраивается в train каждого внешнего блока, 20 trials, одинаковые пациенты и split/seed. Удаление ТАГ/МС включает их собственные индикаторы пропусков; производные ТАГ/МС отсутствуют. Диабет остаётся как компонент метаболического состояния. Результаты не изменяют финальное правило полного набора. Значения улучшения не интерпретируются причинно.

In [ ]:
BRANCHES={"all_real":{"strategy":"all_real","indicators":True,"drop":[]},"class_weights":{"strategy":"class_weights","indicators":True,"drop":[]},"stratified_4to1":{"strategy":"stratified_4to1","indicators":True,"drop":[]},"random_4to1":{"strategy":"random_4to1","indicators":True,"drop":[]},"no_missing_indicators":{"strategy":"all_real","indicators":False,"drop":[]},"without_tag_ms":{"strategy":"all_real","indicators":True,"drop":["tag","ms"]},"tag_only":{"strategy":"all_real","indicators":True,"drop":["ms"]},"ms_only":{"strategy":"all_real","indicators":True,"drop":["tag"]}}
EX_OOF=[];EX_METRICS=[];EX_CONFIG=[]
for repeat,fold,tr,te in OUTER:
    seed=BASE_SEED+100*repeat+10*fold;b=y[tr];g=groups[tr];ss=splits(X.iloc[tr],b,g,seed+1000)
    for branch,settings in BRANCHES.items():
        columns=[c for c in FEATURES if c not in settings["drop"]]
        a=X.iloc[tr][columns]
        result=tune(a,b,g,SELECTED_FAMILY,ss,f"conditional_{branch}_r{repeat}_f{fold}",seed,trials=CONFIG["conditional_trials"],fixed=settings)
        model=train(a,b,SELECTED_FAMILY,result["config"],seed);p=predict(model,X.iloc[te][columns]);t=result["threshold"]
        EX_OOF.extend(records(te,repeat,fold,branch,p,t,result["config"]))
        EX_METRICS.append(dict(repeat=repeat,fold=fold,branch=branch,**metrics(y[te],p,t)))
        EX_CONFIG.append(dict(repeat=repeat,fold=fold,branch=branch,**result))
    pd.DataFrame(EX_OOF).to_csv(OUT/"02_условные_oof.csv",index=False)
    pd.DataFrame(EX_METRICS).to_csv(OUT/"02_условные_метрики_блоков.csv",index=False)
    savejson(EX_CONFIG,"02_условные_конфигурации.json")
    log(f"Conditional outer r{repeat} f{fold} complete")
ex=pd.DataFrame(EX_OOF);em=pd.DataFrame(EX_METRICS)
display(em.groupby("branch")[["AP","ROC_AUC","Brier"]].agg(["mean","std"]))


## Сводки, калибровка и условная неопределённость
Основная сводка — среднее метрик по трём полным OOF-повторениям; дополнительно показаны 9 блоков (SD/диапазон не доверительный интервал). Bootstrap 1000 раз ресэмплирует группы пациентов, сохраняя все их записи и все повторения, а парные сравнения используют одинаковый отбор. Интервалы учитывают условную неопределённость фиксированных OOF-прогнозов, не весь процесс обучения и выбора. Нулевая модель предсказывает частоту смерти своего внешнего train и всех относит к выжившим.

In [ ]:
def summarize_oof(frame,key):
    rows=[]
    for (name,repeat),v in frame.groupby([key,"repeat"]):
        rows.append(dict(name=name,repeat=repeat,**metrics(v.y_true.to_numpy(),v.p_death.to_numpy(),v.threshold.to_numpy())))
    result=pd.DataFrame(rows)
    summary=result.groupby("name").agg(AP=("AP","mean"),AP_repeat_SD=("AP","std"),ROC_AUC=("ROC_AUC","mean"),Brier=("Brier","mean"),sensitivity=("sensitivity","mean"),specificity=("specificity","mean"),precision=("precision","mean"),F1=("F1","mean"),accuracy=("accuracy","mean"),mean_probability=("mean_probability","mean"))
    return result,summary
null=[]
for r,f,tr,te in OUTER:null.extend(records(te,r,f,"prevalence_baseline",np.full(len(te),y[tr].mean()),1.0,{"train_prevalence":float(y[tr].mean())}))
oof=pd.concat([oof,pd.DataFrame(null)],ignore_index=True)
oof.to_csv(OUT/"02_oof_predictions.csv",index=False)
repeats,summary=summarize_oof(oof,"family")
repeats.to_csv(OUT/"02_метрики_повторений.csv",index=False);summary.to_csv(OUT/"02_сравнение_моделей.csv")
exrepeats,exsummary=summarize_oof(ex,"family")
exrepeats.to_csv(OUT/"02_условные_метрики_повторений.csv",index=False);exsummary.to_csv(OUT/"02_условные_сравнения.csv")
# Fixed group resampling, same draws for every family/branch; no pseudoreplication.
unique=np.unique(groups);bygroup={g:np.flatnonzero(groups==g) for g in unique}
rng=np.random.default_rng(BASE_SEED+4444)
BOOT=[np.concatenate([bygroup[g] for g in rng.choice(unique,len(unique),replace=True)]) for _ in range(1000)]
def prediction_arrays(frame,names):
    result={}
    for name in names:
        v=frame[frame.family==name]
        result[name]=np.array([v[v.repeat==r].set_index("row_id").loc[D.row_id,"p_death"].to_numpy() for r in range(3)])
    return result
def boot_scores(arrays):
    scores={name:[] for name in arrays}
    for ids in BOOT:
        if len(np.unique(y[ids]))<2:continue
        for name,p in arrays.items():
            scores[name].append(np.mean([[average_precision_score(y[ids],pr[ids]),roc_auc_score(y[ids],pr[ids]),brier_score_loss(y[ids],pr[ids])] for pr in p],axis=0))
    return {k:np.array(v) for k,v in scores.items()}
bs=boot_scores(prediction_arrays(oof,FAMILIES+["selector"]))
ci=[]
for name,v in bs.items():
    for j,metric in enumerate(["AP","ROC_AUC","Brier"]):
        lo,hi=np.quantile(v[:,j],[.025,.975]);ci.append(dict(family=name,metric=metric,estimate=summary.loc[name,metric],lower=lo,upper=hi,scope="conditional group bootstrap; fixed OOF predictions"))
pd.DataFrame(ci).to_csv(OUT/"02_условные_bootstrap_CI.csv",index=False)
be=boot_scores(prediction_arrays(ex,list(BRANCHES)))
comparisons=[("class_weights","all_real"),("stratified_4to1","all_real"),("random_4to1","all_real"),("all_real","no_missing_indicators"),("all_real","without_tag_ms"),("tag_only","without_tag_ms"),("ms_only","without_tag_ms"),("all_real","tag_only"),("all_real","ms_only")]
deltas=[];fold_deltas=[]
for plus,minus in comparisons:
    for j,metric in enumerate(["AP","ROC_AUC","Brier"]):
        diff=be[plus][:,j]-be[minus][:,j];lo,hi=np.quantile(diff,[.025,.975])
        deltas.append(dict(plus=plus,minus=minus,metric=metric,delta=float(exsummary.loc[plus,metric]-exsummary.loc[minus,metric]),lower=float(lo),upper=float(hi),scope="conditional on globally chosen family and fixed paired OOF"))
    a=em[em.branch==plus].set_index(["repeat","fold"]);b=em[em.branch==minus].set_index(["repeat","fold"])
    for (r,f),v in (a[["AP","ROC_AUC","Brier"]]-b[["AP","ROC_AUC","Brier"]]).iterrows():fold_deltas.append(dict(plus=plus,minus=minus,repeat=r,fold=f,**v.to_dict()))
delta=pd.DataFrame(deltas);delta.to_csv(OUT/"02_парные_изменения.csv",index=False);pd.DataFrame(fold_deltas).to_csv(OUT/"02_парные_изменения_блоков.csv",index=False)
cal=[]
for (family,r),v in pd.concat([oof.assign(kind="main"),ex.assign(kind="conditional")],ignore_index=True).groupby(["family","repeat"]):
    v=v.copy();v["bin"]=pd.qcut(v.p_death,5,duplicates="drop")
    for bn,g in v.groupby("bin",observed=True):cal.append(dict(family=family,repeat=r,bin=str(bn),n=len(g),deaths=int(g.y_true.sum()),mean_prediction=g.p_death.mean(),event_rate=g.y_true.mean()))
cal=pd.DataFrame(cal);cal.to_csv(OUT/"02_калибровка.csv",index=False)
fig,axes=plt.subplots(1,2,figsize=(13,5))
for family in FAMILIES+["selector"]:
    v=cal[cal.family==family]
    axes[0].plot(v.mean_prediction,v.event_rate,"o",alpha=.65,label=family)
    axes[1].bar(family,summary.loc[family,"AP"],color="#35658c")
axes[0].plot([0,1],[0,1],"--",color="grey");axes[0].set(xlabel="Средний прогноз смерти",ylabel="Частота смерти",title="Калибровка: 5 квантильных групп × 3 повторения");axes[0].legend()
axes[1].axhline(y.mean(),ls="--",color="grey",label="Частота смерти");axes[1].set(ylabel="AP смерти",title="Средний AP полных внешних OOF-повторений");axes[1].legend();figsave("02_качество_и_калибровка")
for family in FAMILIES:
    plt.figure(figsize=(6,5));v=cal[cal.family==family]
    for r,g in v.groupby("repeat"):plt.plot(g.mean_prediction,g.event_rate,"o-",label=f"Повторение {r+1}")
    plt.plot([0,1],[0,1],"--",color="grey");plt.xlabel("Прогноз смерти");plt.ylabel("Частота смерти");plt.title(family+": калибровка на реальных внешних пациентах");plt.legend();figsave("02_калибровка_"+family)
plt.figure(figsize=(10,5));em.groupby("branch").AP.mean().plot.bar(color="#35658c");plt.ylabel("AP: среднее внешних блоков");plt.title("Условное сравнение обработки и ТАГ/МС");figsave("02_условные_эксперименты")
trials_frames=[]
for study in optuna.get_all_study_summaries(storage=STORAGE):
    if study.study_name.startswith(RUN_ID):
        s=optuna.load_study(study_name=study.study_name,storage=STORAGE);t=s.trials_dataframe();t.insert(0,"study",study.study_name);trials_frames.append(t)
trials_df=pd.concat(trials_frames,ignore_index=True);trials_df.to_csv(OUT/"02_trials.csv",index=False)
pd.DataFrame(SAMPLING_AUDIT).to_csv(OUT/"02_аудит_балансировки.csv",index=False)
savejson({"run_id":RUN_ID,"protocol":CONFIG,"data_sha256":sha(OUT/"01_данные.csv"),"model_sha256":sha(OUT/"02_final_model.joblib"),"studies":len(trials_frames),"trial_states":trials_df.state.value_counts().to_dict(),"completed_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()},"02_происхождение_результатов.json")
log("All nested, final, controlled experiments completed")
display(summary);display(exsummary);display(delta)
